# Phase 4: Supervised Surface Defect Classification (Google Colab GPU)

**Objective:** Train and benchmark three classification models on the NEU Surface Defect dataset using a Tesla T4 GPU:
1. **SimpleCNN** (Trained from scratch — baseline)
2. **ResNet-18** (Transfer learning with residual skip connections)
3. **EfficientNet-B0** (Transfer learning with compound scaling & depthwise separable convolutions)

### Important Architecture Role Clarification:
The NEU dataset contains **6 defect categories**: `crazing`, `inclusion`, `patches`, `pitted_surface`, `rolled-in_scale`, `scratches`.
This classifier answers: **"Which defect type is present on this defective surface?"**
In our production quality inspection system, this serves as **Tier 2 (Root Cause & Defect Categorization)**, working alongside **Tier 1 (PatchCore Anomaly Detection)** which answers **"Is this product normal or defective?"**

## Step 1: Verify Google Colab GPU (Tesla T4)

Ensure you have selected **Runtime -> Change runtime type -> T4 GPU** before executing.

In [ ]:
import torch
import sys

print(f"Python Version:  {sys.version.split()[0]}")
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available:  {torch.cuda.is_available()}")

if not torch.cuda.is_available():
    raise RuntimeError("GPU not detected! Please enable GPU in Colab: Runtime -> Change runtime type -> T4 GPU")

print(f"GPU Allocated:   {torch.cuda.get_device_name(0)}")
print(f"Available VRAM:  {torch.cuda.get_device_properties(0).total_mem / (1024**3):.1f} GB")

## Step 2: Install Dependencies and Clone / Mount Codebase

In [ ]:
# Install lightweight requirements
!pip install -q huggingface_hub torchvision torchmetrics scikit-learn matplotlib pandas

# Set seed for reproducibility
torch.manual_seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(42)

## Step 3: Download and Load NEU Surface Defect Dataset

We load the verified `KeenForgeAI/NEU-DET-corrected` dataset from Hugging Face.

In [ ]:
from huggingface_hub import snapshot_download
from pathlib import Path

DATA_DIR = Path("./neu_surface")
DATA_DIR.mkdir(parents=True, exist_ok=True)

print("Downloading NEU Surface Defect Dataset...")
snapshot_download(
    repo_id="KeenForgeAI/NEU-DET-corrected",
    repo_type="dataset",
    local_dir=str(DATA_DIR)
)
print("Download complete!")

## Step 4: Run Training Script Across All 3 Architectures

This executes the benchmark harness: SimpleCNN vs ResNet-18 vs EfficientNet-B0.

In [ ]:
# Execute the training harness
!python training/colab/classification/03_train_classification_models.py

## Step 5: Display Comparative Performance Results and Confusion Matrices

In [ ]:
import pandas as pd
from PIL import Image
import matplotlib.pyplot as plt

results_csv = Path("./docs/assets/models/classification_model_comparison.csv")
if results_csv.exists():
    df = pd.read_csv(results_csv)
    display(df)

# Display Confusion Matrices
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
models = ["SimpleCNN_Baseline", "ResNet18_Transfer", "EfficientNetB0_Transfer"]

for idx, m in enumerate(models):
    p = Path(f"./docs/assets/models/confusion_matrix_{m}.png")
    if p.exists():
        axes[idx].imshow(Image.open(p))
        axes[idx].set_title(m)
        axes[idx].axis("off")

plt.tight_layout()
plt.show()